# News Topic Classifier — Analysis Notebook

This notebook walks through:
1. Data loading and exploration
2. Preprocessing demo
3. TF-IDF model training
4. Evaluation and visualisation

## 1. Data Loading & Exploration

In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('../data/sample_data.csv')
print(f'Total rows: {len(df)}')
print(f'Columns: {list(df.columns)}')
df.head()

In [ ]:
# Class distribution
counts = df['category'].value_counts()
print(counts)

fig, ax = plt.subplots(figsize=(9, 4))
counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_title('Articles per Category')
ax.set_xlabel('Category')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.savefig('../outputs/category_distribution.png', dpi=150)
plt.show()
print('Saved category_distribution.png')

In [ ]:
# Text length distribution
df['text_length'] = df['text'].str.len()
print(df['text_length'].describe())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(df['text_length'], bins=30, color='teal', edgecolor='white')
ax.set_title('Distribution of Article Text Lengths')
ax.set_xlabel('Character count')
ax.set_ylabel('Frequency')
plt.tight_layout()
plt.savefig('../outputs/text_length_distribution.png', dpi=150)
plt.show()

## 2. Preprocessing Demo

In [ ]:
from src.preprocess import TextPreprocessor

preprocessor = TextPreprocessor()

samples = [
    "Breaking News! Visit http://example.com — The <b>stock market</b> rose 3% today.",
    df['text'].iloc[0],
    df['text'].iloc[50],
]

for s in samples:
    print('ORIGINAL :', s[:100])
    print('PROCESSED:', preprocessor.preprocess(s)[:100])
    print()

## 3. TF-IDF Model Training

In [ ]:
import os
os.makedirs('../outputs', exist_ok=True)
os.makedirs('../models', exist_ok=True)

from src.train_tfidf import train_and_evaluate
model_data = train_and_evaluate()
print(f"Best model: {model_data['best_model_name']}")

## 4. Evaluation & Visualisation

In [ ]:
from IPython.display import Image, display

cm_path = '../outputs/tfidf_confusion_matrix.png'
if os.path.exists(cm_path):
    display(Image(cm_path))
else:
    print('Run training first to generate confusion matrix.')

In [ ]:
# Interactive prediction demo
import joblib

data = joblib.load('../models/tfidf_model.pkl')
pipeline = data['pipeline']
prep = data['preprocessor']
label_list = data['label_list']

test_articles = [
    "NASA discovers a new exoplanet with potential signs of liquid water in its atmosphere.",
    "The Federal Reserve raised interest rates by 25 basis points citing persistent inflation.",
    "The championship team celebrated with a parade through downtown after winning the title.",
    "A new blockbuster film broke box office records in its opening weekend globally.",
]

for article in test_articles:
    pred = pipeline.predict([prep.preprocess(article)])[0]
    print(f'[{pred:>14}]  {article[:75]}…')